In [ ]:
import requests
import csv
import os
import threading
import urllib3
from concurrent.futures import ThreadPoolExecutor, as_completed

# Tắt cảnh báo SSL rác màn hình (nếu dùng verify=False)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

lock = threading.Lock()
completed = 0
success_count = 0
fail_count = 0

CSV_FILE = "deface_labels.csv"
# Đổi đường dẫn input theo file đã phân loại
INPUT_FILE = "Classified_URLs/urls_web.txt"


def load_urls(file_path):
    if not os.path.exists(file_path):
        print(f"[!] Không tìm thấy file: {file_path}")
        return []
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def create_filename(url):
    filename = url.replace("http://", "").replace("https://", "")
    filename = filename.replace("/", "_")
    filename = filename.replace("?", "_").replace("&", "_").replace("=", "_")
    return filename[:200] + ".html"


def fetch_html(url, retries=3):
    for _ in range(retries):
        try:
            # Vẫn khuyến nghị thêm verify=False để không bỏ lỡ các trang bị lỗi chứng chỉ SSL
            res = requests.get(url, headers=HEADERS, timeout=(3, 5), verify=False)
            if res.status_code == 200 and res.text:
                return url, res.text
        except:
            pass
    return url, None


def load_existing_urls():
    if not os.path.exists(CSV_FILE):
        return set()

    urls = set()
    with open(CSV_FILE, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if row:
                urls.add(row[0])
    return urls


# Hàm mới: Chỉ tạo file và ghi Header 1 lần duy nhất lúc chạy chương trình
def init_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["URL", "HTML_File_Name", "Label"])


def append_csv(row):
    # Đã có init_csv lo việc tạo file, giờ chỉ việc append (ghi tiếp)
    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(row)


def save_result(result, total, existing_urls):
    global completed, success_count, fail_count

    url, html = result

    with lock:
        completed += 1
        percent = (completed / total) * 100

    if not html:
        with lock:
            fail_count += 1
        print(f"[{completed}/{total} - {percent:.1f}%] FAILED: {url}")
        return None

    filename = create_filename(url)
    path = os.path.join("html_pages", filename)

    # Đã thêm errors="ignore" để tránh lỗi Unicode làm sập thread
    try:
        with open(path, "w", encoding="utf-8", errors="ignore") as f:
            f.write(html)
    except Exception as e:
        print(f"[{completed}/{total} - {percent:.1f}%] LỖI GHI FILE: {url} - {e}")
        return None

    # 🔥 Critical section: check + write + update state
    with lock:
        if url in existing_urls:
            print(f"[SKIP] {url}")
            return None

        append_csv([url, filename, "unlabeled"])
        existing_urls.add(url)
        success_count += 1

    print(f"[{completed}/{total} - {percent:.1f}%] OK: {url}")
    return True


def main():
    urls = load_urls(INPUT_FILE)
    total = len(urls)
    
    if total == 0:
        return

    os.makedirs("html_pages", exist_ok=True)

    # Chạy khởi tạo CSV trước khi bắt đầu đa luồng
    init_csv()

    existing_urls = load_existing_urls()

    num_threads = 10

    print(f"Total URLs: {total}")
    print(f"Running with {num_threads} threads...")
    print("CSV PATH:", os.path.abspath(CSV_FILE), "\n")

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(fetch_html, url) for url in urls]

        for future in as_completed(futures):
            save_result(future.result(), total, existing_urls)

    print("\n===== SUMMARY =====")
    print(f"Total: {total}")
    print(f"Success: {success_count}")
    print(f"Failed: {fail_count}")
    print(f"CSV rows (approx): {len(existing_urls)}")

    print("\nDone!")


if __name__ == "__main__":
    main()